In [2]:
import json
import random
import nibabel as nib
import pandas as pd
from pathlib import Path

# 设置随机种子，确保结果可复现
random.seed(42)

# 原有函数：构建 Subject -> Group 映射
def build_subject_group_map(csv_path, subject_col='Subject', group_col='Group'):
    df = pd.read_csv(csv_path)
    dup_check = df.groupby(subject_col)[group_col].nunique()
    inconsistent = dup_check[dup_check > 1].index.tolist()
    if inconsistent:
        raise ValueError(f"以下 Subject 对应多个不同 Group: {inconsistent}")
    subject_group = df[[subject_col, group_col]].drop_duplicates(subset=[subject_col])
    return dict(zip(subject_group[subject_col], subject_group[group_col]))

# 原有函数：收集所有样本（CN 和 MCI）
def get_samples(data_dir, subject_group_map):
    results = []
    data_dir = Path(data_dir)
    num_cn = 0
    num_mci = 0
    for nii_file in data_dir.glob("*.nii*"):
        img = nib.load(str(nii_file))
        subject = str(img.header['db_name']).split("'")[1]
        group = subject_group_map.get(subject)
        if group is None:
            assert False
        if group == "AD":
            continue
        sample = {}
        sample["images"] = [str(nii_file)]
        sample["label"] = 0 if group == "CN" else 1
        if sample['label'] == 0:
            num_cn += 1
        else:
            num_mci += 1
        results.append(sample)
    print(f"原始 CN 样本数: {num_cn}, MCI 样本数: {num_mci}")
    return results


csv_path = "/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/ADNI1_Complete_1Yr_3T_3_23_2026.csv"
data_root = "/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/ADNI_1year_3T_gz"

subject_group = build_subject_group_map(csv_path)
samples = get_samples(data_root, subject_group)

samples_0 = [s for s in samples if s['label'] == 0]
samples_1 = [s for s in samples if s['label'] == 1]

min_count = min(len(samples_0), len(samples_1))
print(f"平衡后每类样本数: {min_count}")

balanced_0 = random.sample(samples_0, min_count)
balanced_1 = random.sample(samples_1, min_count)

balanced_samples = balanced_0 + balanced_1
random.shuffle(balanced_samples)

split_ratio = 0.7
split_idx = int(len(balanced_samples) * split_ratio)
train_samples = balanced_samples[:split_idx]
test_samples = balanced_samples[split_idx:]

print(f"训练集大小: {len(train_samples)}, 测试集大小: {len(test_samples)}")

# --- 保存为 JSON ---
data = {
    "train": train_samples,
    "val": test_samples  
}

output_path = Path("/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/adni.json")
output_path.write_text(
    json.dumps(data, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

原始 CN 样本数: 121, MCI 样本数: 173
平衡后每类样本数: 121
训练集大小: 169, 测试集大小: 73


48676

In [1]:
import os
import json
import random
from pathlib import Path



def main(data_root, output_path, ratio=0.7):
    ncs, mcis = [], []
    root = Path(data_root)
    for file in root.glob("NC*"):
        image = os.path.join(str(file),os.listdir(str(file))[-1])
    
        sample = {
            "images":[image],
            "label": 0
        }
        ncs.append(sample)

    for file in root.glob("*MCI*"):
        image = os.path.join(str(file),os.listdir(str(file))[-1])
    
        sample = {
            "images":[image],
            "label": 1
        }
        mcis.append(sample)

    count = min(len(ncs), len(mcis))
    bncs = random.sample(ncs, count)
    bmcis = random.sample(mcis, count)
    
    train_num = int(ratio*count)
    train_samples = bncs[: train_num]+bmcis[: train_num]
    val_samples = bncs[train_num: ]+ bmcis[train_num: ]
    random.shuffle(train_samples)
    random.shuffle(val_samples)
    data = {
        "train": train_samples,
        "val": val_samples
    }
    output_path = Path(output_path)
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
data_root = "/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/BrainMVP-ds"
output_path = "/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/brainmvp_adni.json"
main(data_root, output_path)
